# Model comparison: feedforward NN vs GCN vs GAT


This notebook compares all classification models trained in this project:
- Feedforward Neural Network (NN)
- Graph Convolutional Network (GCN)
- Graph Attention Network (GAT)

Each model is trained and evaluated on SM vs each of the 6 BSM SMEFT operators:
**cbbim, cbgim, cbhim, chbtil, chgtil, ctbim**

Metrics compared:
- Training time
- Inference time
- Classification accuracy, precision, recall, F1, AUC

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from plot_style import apply_publication_style
apply_publication_style()
import time
import os
import h5py
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import Data, DataLoader as GeoDataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool, global_max_pool, BatchNorm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

os.makedirs('plots', exist_ok=True)
os.makedirs('metrics', exist_ok=True)

## 1. Load Data

In [ ]:
def load_dataset(filepath):
    with h5py.File(filepath, 'r') as f:
        df = pd.DataFrame.from_records(f['ForAnalysis/1d'][:])
    return df

sm_df = load_dataset('datasets/new_Input_bbyy_SMEFT_SM_4thMarch_2026.h5')
cbbim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_cbbim_4thMarch_2026.h5')
cbgim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_cbgim_4thMarch_2026.h5')
cbhim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_cbhim_4thMarch_2026.h5')
chbtil_df = load_dataset('datasets/new_Input_bbyy_SMEFT_chbtil_4thMarch_2026.h5')
chgtil_df = load_dataset('datasets/new_Input_bbyy_SMEFT_chgtil_4thMarch_2026.h5')
ctbim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_ctbim_4thMarch_2026.h5')

bsm_datasets = {
    'cbbim': cbbim_df,
    'cbgim': cbgim_df,
    'cbhim': cbhim_df,
    'chbtil': chbtil_df,
    'chgtil': chgtil_df,
    'ctbim': ctbim_df,
}

exclude_columns = ['EventNumber', 'is_HHEvent', 'is_SingleHiggsEvent', 'is_SingleZEvent', 'is_ZHEvent', 'Lumi_weight', 'nBTaggedJets', 'NJets']
feature_columns = [col for col in sm_df.columns if col not in exclude_columns]

print(f"Features ({len(feature_columns)}): {feature_columns}")
print(f"SM: {len(sm_df):,}")
for name, df in bsm_datasets.items():
    print(f"  {name}: {len(df):,}")

## 2. Define All Models

In [ ]:
# Feedforward NN Classifier
class NNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout=0.3):
        super(NNClassifier, self).__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        layers.append(nn.Sigmoid())
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x).squeeze()

# GCN Classifier
class GCNClassifier(nn.Module):
    def __init__(self, node_features=4, global_features=10, hidden_dim=64, num_layers=3, dropout=0.3):
        super(GCNClassifier, self).__init__()
        self.num_layers = num_layers
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        self.convs.append(GCNConv(node_features, hidden_dim))
        self.bns.append(BatchNorm(hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.bns.append(BatchNorm(hidden_dim))
        
        combined_dim = hidden_dim * 2 + global_features
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
        self.dropout = dropout
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for i in range(self.num_layers):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        global_features = data.u.view(-1, 10)
        combined = torch.cat([x_mean, x_max, global_features], dim=1)
        return self.classifier(combined).squeeze()

# GAT Classifier
class GATClassifier(nn.Module):
    def __init__(self, node_features=4, global_features=10, hidden_dim=32, num_layers=3, heads=4, dropout=0.3):
        super(GATClassifier, self).__init__()
        self.num_layers = num_layers
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        self.convs.append(GATConv(node_features, hidden_dim, heads=heads, dropout=dropout))
        self.bns.append(BatchNorm(hidden_dim * heads))
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_dim * heads, hidden_dim, heads=heads, dropout=dropout))
            self.bns.append(BatchNorm(hidden_dim * heads))
        self.convs.append(GATConv(hidden_dim * heads, hidden_dim, heads=1, concat=False, dropout=dropout))
        self.bns.append(BatchNorm(hidden_dim))
        
        combined_dim = hidden_dim * 2 + global_features
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
        self.dropout = dropout
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for i in range(self.num_layers):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        global_features = data.u.view(-1, 10)
        combined = torch.cat([x_mean, x_max, global_features], dim=1)
        return self.classifier(combined).squeeze()

## 3. Prepare Data

In [ ]:
def _get_event_type(row):
    if row.get('is_ZHEvent', False):
        return 'ZH'
    if row.get('is_HHEvent', False):
        return 'HH'
    if row.get('is_SingleHiggsEvent', False):
        return 'Single H'
    return 'Other'

def _stratified_sample_sm_bsm(sm_df, bsm_df, random_state=42):
    sm_copy = sm_df.copy()
    bsm_copy = bsm_df.copy()
    sm_copy['_event_type'] = sm_copy.apply(_get_event_type, axis=1)
    bsm_copy['_event_type'] = bsm_copy.apply(_get_event_type, axis=1)
    sm_samples, bsm_samples = [], []
    for etype in ['ZH', 'HH', 'Single H', 'Other']:
        sm_sub = sm_copy[sm_copy['_event_type'] == etype]
        bsm_sub = bsm_copy[bsm_copy['_event_type'] == etype]
        n_min = min(len(sm_sub), len(bsm_sub))
        if n_min > 0:
            sm_samples.append(sm_sub.sample(n=n_min, random_state=random_state))
            bsm_samples.append(bsm_sub.sample(n=n_min, random_state=random_state))
    if sm_samples and bsm_samples:
        return pd.concat(sm_samples, ignore_index=True), pd.concat(bsm_samples, ignore_index=True)
    n_min = min(len(sm_df), len(bsm_df))
    return sm_df.sample(n=n_min, random_state=random_state), bsm_df.sample(n=n_min, random_state=random_state)

def prepare_nn_data(sm_df, bsm_df, feature_columns):
    """Prepare balanced NN data for SM vs one BSM operator. Stratified by event type."""
    sm_sample, bsm_sample = _stratified_sample_sm_bsm(sm_df, bsm_df)
    n_min = len(sm_sample)

    X = np.vstack([sm_sample[feature_columns].values, bsm_sample[feature_columns].values])
    y = np.concatenate([np.zeros(len(sm_sample)), np.ones(len(bsm_sample))])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, y_train, y_test, sm_sample, bsm_sample

print("NN data preparation function defined.")

In [ ]:
def event_to_graph(row, label):
    node_features = torch.tensor([
        [row['Higgs_pT'], row['Higgs_Eta'], row['Higgs_Phi'], row['Higgs_Mass']],
        [row['LeadJet_pT'], row['LeadJet_Eta'], row['LeadJet_Phi'], row['LeadJet_M']],
        [row['SubLeadJet_pT'], row['SubLeadJet_Eta'], row['SubLeadJet_Phi'], row['SubLeadJet_M']],
        [row['LeadPhoton_pT'], row['LeadPhoton_Eta'], row['LeadPhoton_Phi'], 0.0],
        [row['SubLeadPhoton_pT'], row['SubLeadPhoton_Eta'], row['SubLeadPhoton_Phi'], 0.0],
    ], dtype=torch.float)
    
    edge_index = []
    for i in range(5):
        for j in range(5):
            if i != j:
                edge_index.append([i, j])
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    
    global_features = torch.tensor([
        row['m_bbyy'], row['pT_jj'], row['Eta_jj'], row['DPhi_bb'],
        row['signed_DeltaPhi_jj'], row['cosThetaStar'], row['costheta1'], row['costheta2'],
        row['DPhi_yybb'], row['Eta_yybb'],
    ], dtype=torch.float)
    
    return Data(x=node_features, edge_index=edge_index, u=global_features, y=torch.tensor([label], dtype=torch.float))

def prepare_gnn_data(sm_sample, bsm_sample):
    """Convert balanced SM/BSM samples to graph datasets."""
    graphs = []
    for _, row in sm_sample.iterrows():
        graphs.append(event_to_graph(row, 0))
    for _, row in bsm_sample.iterrows():
        graphs.append(event_to_graph(row, 1))
    np.random.seed(42)
    np.random.shuffle(graphs)
    n_train = int(0.8 * len(graphs))
    return graphs[:n_train], graphs[n_train:]

print("GNN data preparation function defined.")

## 4. Train All Models

In [ ]:
def train_nn(model, X_train, y_train, epochs=50, batch_size=256, lr=0.001):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    metrics = {'loss': [], 'acc': [], 'epoch_time': []}
    start = time.time()
    
    for epoch in range(epochs):
        epoch_start = time.time()
        model.train()
        losses, correct, total = [], 0, 0
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(X_batch)
            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            correct += ((out > 0.5).float() == y_batch).sum().item()
            total += y_batch.size(0)
        
        metrics['loss'].append(np.mean(losses))
        metrics['acc'].append(correct / total)
        metrics['epoch_time'].append(time.time() - epoch_start)
        
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d} | Loss: {metrics['loss'][-1]:.4f} | Acc: {metrics['acc'][-1]:.4f}")
    
    return metrics, time.time() - start

def train_gnn(model, train_graphs, epochs=50, batch_size=64, lr=0.001):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    dataloader = GeoDataLoader(train_graphs, batch_size=batch_size, shuffle=True)
    
    metrics = {'loss': [], 'acc': [], 'epoch_time': []}
    start = time.time()
    
    for epoch in range(epochs):
        epoch_start = time.time()
        model.train()
        losses, correct, total = [], 0, 0
        for batch in dataloader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch)
            loss = criterion(out, batch.y.float())
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            correct += ((out > 0.5).float() == batch.y).sum().item()
            total += batch.num_graphs
        
        metrics['loss'].append(np.mean(losses))
        metrics['acc'].append(correct / total)
        metrics['epoch_time'].append(time.time() - epoch_start)
        
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d} | Loss: {metrics['loss'][-1]:.4f} | Acc: {metrics['acc'][-1]:.4f}")
    
    return metrics, time.time() - start

In [ ]:
all_results = {}

for bsm_name, bsm_df in bsm_datasets.items():
    print(f"\n{'='*70}")
    print(f"  BSM Operator: {bsm_name}")
    print(f"{'='*70}")
    
    # Prepare data
    X_train_scaled, X_test_scaled, y_train, y_test, sm_sample, bsm_sample = \
        prepare_nn_data(sm_df, bsm_df, feature_columns)
    train_graphs, test_graphs = prepare_gnn_data(sm_sample, bsm_sample)
    print(f"  NN data — Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")
    print(f"  GNN data — Train: {len(train_graphs)}, Test: {len(test_graphs)}")
    
    # Train NN
    print(f"\n  Training Feedforward NN...")
    nn_model = NNClassifier(len(feature_columns)).to(device)
    nn_metrics, nn_train_time = train_nn(nn_model, X_train_scaled, y_train, epochs=50)
    print(f"  NN Training time: {nn_train_time:.2f}s")
    
    # Train GCN
    print(f"\n  Training GCN...")
    gcn_model = GCNClassifier().to(device)
    gcn_metrics, gcn_train_time = train_gnn(gcn_model, train_graphs, epochs=50)
    print(f"  GCN Training time: {gcn_train_time:.2f}s")
    
    # Train GAT
    print(f"\n  Training GAT...")
    gat_model = GATClassifier().to(device)
    gat_metrics, gat_train_time = train_gnn(gat_model, train_graphs, epochs=50)
    print(f"  GAT Training time: {gat_train_time:.2f}s")
    
    all_results[bsm_name] = {
        'nn': {'model': nn_model, 'metrics': nn_metrics, 'train_time': nn_train_time,
               'X_test': X_test_scaled, 'y_test': y_test},
        'gcn': {'model': gcn_model, 'metrics': gcn_metrics, 'train_time': gcn_train_time,
                'test_graphs': test_graphs},
        'gat': {'model': gat_model, 'metrics': gat_metrics, 'train_time': gat_train_time,
                'test_graphs': test_graphs},
    }

print(f"\nDone training all models for {len(bsm_datasets)} BSM operators.")

In [ ]:
# (Training is handled in the loop above)

In [ ]:
# (Training is handled in the loop above)

## 5. Evaluate All Models

In [ ]:
def evaluate_nn(model, X_test, y_test):
    model.eval()
    X_test_t = torch.FloatTensor(X_test).to(device)
    start = time.time()
    with torch.no_grad():
        probs = model(X_test_t).cpu().numpy()
    inf_time = time.time() - start
    
    preds = (probs > 0.5).astype(int)
    fpr, tpr, _ = roc_curve(y_test, probs)
    return {
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds),
        'recall': recall_score(y_test, preds),
        'f1': f1_score(y_test, preds),
        'auc': auc(fpr, tpr),
        'inference_time': inf_time,
        'events_per_second': len(y_test) / inf_time,
        'probs': probs, 'fpr': fpr, 'tpr': tpr, 'labels': y_test
    }

def evaluate_gnn(model, test_graphs):
    model.eval()
    dataloader = GeoDataLoader(test_graphs, batch_size=64, shuffle=False)
    probs, labels = [], []
    start = time.time()
    with torch.no_grad():
        for batch in dataloader:
            batch = batch.to(device)
            probs.extend(model(batch).cpu().numpy())
            labels.extend(batch.y.cpu().numpy())
    inf_time = time.time() - start
    
    probs, labels = np.array(probs), np.array(labels)
    preds = (probs > 0.5).astype(int)
    fpr, tpr, _ = roc_curve(labels, probs)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds),
        'recall': recall_score(labels, preds),
        'f1': f1_score(labels, preds),
        'auc': auc(fpr, tpr),
        'inference_time': inf_time,
        'events_per_second': len(labels) / inf_time,
        'probs': probs, 'fpr': fpr, 'tpr': tpr, 'labels': labels
    }

all_evals = {}
for bsm_name, res in all_results.items():
    print(f"\nEvaluating models for SM vs {bsm_name}...")
    nn_eval = evaluate_nn(res['nn']['model'], res['nn']['X_test'], res['nn']['y_test'])
    gcn_eval = evaluate_gnn(res['gcn']['model'], res['gcn']['test_graphs'])
    gat_eval = evaluate_gnn(res['gat']['model'], res['gat']['test_graphs'])
    
    all_evals[bsm_name] = {'nn': nn_eval, 'gcn': gcn_eval, 'gat': gat_eval}
    
    print(f"  NN:  AUC={nn_eval['auc']:.4f}, Acc={nn_eval['accuracy']:.4f}, {nn_eval['events_per_second']:,.0f} events/s")
    print(f"  GCN: AUC={gcn_eval['auc']:.4f}, Acc={gcn_eval['accuracy']:.4f}, {gcn_eval['events_per_second']:,.0f} events/s")
    print(f"  GAT: AUC={gat_eval['auc']:.4f}, Acc={gat_eval['accuracy']:.4f}, {gat_eval['events_per_second']:,.0f} events/s")

## 6. Comparison Visualizations

In [ ]:
bsm_names = list(all_evals.keys())
n_ops = len(bsm_names)

# --- Figure 1: ROC curves per operator ---
n_cols = 3
n_rows = int(np.ceil(n_ops / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = np.atleast_2d(axes)

for idx, bsm_name in enumerate(bsm_names):
    ax = axes[idx // n_cols, idx % n_cols]
    evals = all_evals[bsm_name]
    for model_name, style in [('nn', '-'), ('gcn', '--'), ('gat', ':')]:
        ev = evals[model_name]
        label = f"{model_name.upper()} (AUC={ev['auc']:.4f})"
        ax.plot(ev['fpr'], ev['tpr'], style, label=label, linewidth=2)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title(f'SM vs {bsm_name}', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

for idx in range(n_ops, n_rows * n_cols):
    axes[idx // n_cols, idx % n_cols].set_visible(False)

plt.suptitle('ROC Curves: NN vs GCN vs GAT per BSM Operator', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('plots/model_comparison_roc.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Figure 2: AUC comparison across operators ---
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(n_ops)
width = 0.25
for i, (model_name, color) in enumerate([('nn', '#1f77b4'), ('gcn', '#ff7f0e'), ('gat', '#2ca02c')]):
    aucs = [all_evals[b][model_name]['auc'] for b in bsm_names]
    ax.bar(x + i * width, aucs, width, label=model_name.upper(), color=color, alpha=0.8)

ax.set_xlabel('BSM Operator', fontsize=12)
ax.set_ylabel('AUC', fontsize=12)
ax.set_title('AUC Comparison: NN vs GCN vs GAT across BSM Operators', fontsize=14)
ax.set_xticks(x + width)
ax.set_xticklabels(bsm_names)
ax.set_ylim(0.5, 1.0)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('plots/model_comparison_auc.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Figure 3: Training time comparison ---
fig, ax = plt.subplots(figsize=(10, 5))
for i, (model_name, color) in enumerate([('nn', '#1f77b4'), ('gcn', '#ff7f0e'), ('gat', '#2ca02c')]):
    times = [all_results[b][model_name]['train_time'] for b in bsm_names]
    ax.bar(x + i * width, times, width, label=model_name.upper(), color=color, alpha=0.8)

ax.set_xlabel('BSM Operator', fontsize=12)
ax.set_ylabel('Training Time (s)', fontsize=12)
ax.set_title('Training Time: NN vs GCN vs GAT across BSM Operators', fontsize=14)
ax.set_xticks(x + width)
ax.set_xticklabels(bsm_names)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('plots/model_comparison_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary Table

In [ ]:
rows = []
for bsm_name in bsm_names:
    res = all_results[bsm_name]
    evals = all_evals[bsm_name]
    for model_key, model_label in [('nn', 'NN'), ('gcn', 'GCN'), ('gat', 'GAT')]:
        ev = evals[model_key]
        n_params = sum(p.numel() for p in res[model_key]['model'].parameters())
        rows.append({
            'BSM Operator': bsm_name,
            'Model': model_label,
            'Parameters': n_params,
            'Training Time (s)': f"{res[model_key]['train_time']:.2f}",
            'Accuracy': f"{ev['accuracy']:.4f}",
            'Precision': f"{ev['precision']:.4f}",
            'Recall': f"{ev['recall']:.4f}",
            'F1 Score': f"{ev['f1']:.4f}",
            'AUC': f"{ev['auc']:.4f}",
            'Inference (events/s)': f"{ev['events_per_second']:,.0f}",
        })

comparison_df = pd.DataFrame(rows)

print("\n" + "=" * 90)
print("MODEL COMPARISON SUMMARY — All BSM Operators")
print("=" * 90)
for bsm_name in bsm_names:
    sub = comparison_df[comparison_df['BSM Operator'] == bsm_name]
    print(f"\n--- SM vs {bsm_name} ---")
    print(sub.drop(columns='BSM Operator').to_string(index=False))

comparison_df.to_csv('metrics/model_comparison.csv', index=False)
print(f"\nSaved to metrics/model_comparison.csv ({len(comparison_df)} rows)")